In [ ]:
# pip install timm pandas scikit-learn torch torchvision

import timm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from sklearn.metrics import accuracy_score
import time

class FastDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row.image_path).convert('RGB')
        if self.transform: img = self.transform(img)
        label = row.label if 'label' in row else -1
        return img, torch.tensor(label, dtype=torch.long)

# ------------------- САМЫЕ БЫСТРЫЕ И СИЛЬНЫЕ модели 2025 (инференс < 1 сек на 1000 фото на одной 4090/A100) -------------------
fast_top_models = [
    "eva02_large_patch14_448.mim_in22k_ft_in22k_in1k",     # №1 на многих соревах 2024-2025
    "convnextv2_large.fcmae_ft_in22k_in1k_384",           # очень быстро + топ
    "beitv2_large_patch16_224.in22k_ft_in22k_in1k",       # стабильно топ-3
    "swin_v2_cr_large_384",                               # новый swin 2025 — огонь
    "fastvit_ma36.SAMViT_patch16_224",                    # САМЫЙ БЫСТРЫЙ из топовых (инференс в 2–3 раза быстрее ViT)
    "tiny_vit_21m_512.dist_in22k_ft_in1k",                # если совсем мало времени (но всё ещё > ResNet50)
]

model_name = "fastvit_ma36.SAMViT_patch16_224"  # ← мой личный фаворит на 2025 соревах

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # или 384/448 если модель поддерживает
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = FastDataset(train_df, transform)
test_ds  = FastDataset(test_df,  transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(model_name, pretrained=True, num_classes=train_df['label'].nunique())
model = model.to(device)

# Если времени совсем мало — просто замораживаем backbone и учим только голову (5–10 минут на 100k фото)
for param in model.parameters():
    param.requires_grad = False
for param in model.get_classifier().parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.get_classifier().parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Обучение 3–10 эпох (обычно хватает)
for epoch in range(5):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        loss = criterion(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Инференс
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())

submission['label'] = preds
submission.to_csv('sub_timm_fast.csv', index=False)

In [ ]:
# pip install sentence-transformers scikit-learn lightgbm xgboost

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import numpy as np

# ------------------- САМЫЕ БЫСТРЫЕ РУССКИЕ/МУЛЬТИЯЗЫЧНЫЕ модели 2025 -------------------
fast_text_models = [
    "intfloat(multilingual-e5-small)",      # 2025 лидер по скорости (инференс 384 токена за 1–2 мс)
    "intfloat(multilingual-e5-base)",       # чуть медленнее, но сильнее
    "BAAI/bge-m3",                          # плотные + sparse + colbert — универсальный монстр
    "cointegrated/rubert-tiny2-sentence",   # чисто русский, ультрабыстрый
    "sentence-transformers/all-MiniLM-L6-v2",  # если только английский
]

model_st = SentenceTransformer("intfloat/multilingual-e5-small")  # мой выбор на 99% соревах

# Эмбеддинги за секунды
train_texts = train_df['text'].tolist()  # или train_df['title'] + ' ' + train_df['description']
test_texts  = test_df['text'].tolist()

X_train = model_st.encode(train_texts, batch_size=512, show_progress_bar=True, normalize_embeddings=True)
X_test  = model_st.encode(test_texts,  batch_size=512, show_progress_bar=True, normalize_embeddings=True)

y_train = train_df['label']

# Вариант A: просто логистическая регрессия (инференс мгновенный)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

# Вариант B: LightGBM/XGBoost (чаще даёт +0.01–0.03)
clf = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, max_depth=-1)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

submission['label'] = preds
submission.to_csv('sub_st_fast.csv', index=False)

In [ ]:
# Просто конкатенируем эмбеддинги
text_emb = model_st.encode(texts)
img_emb  = [] 
with torch.no_grad():
    for batch in img_loader:
        img_emb.extend(model_img(batch[0].to(device)).cpu().numpy())

tabular_features = train_df.drop(['text', 'image_path', 'title', 'description'], axis=1)

X = np.hstack([text_emb, np.array(img_emb), tabular_features.values])
# дальше LightGBM / CatBoost / NN — и ты в топ-3 на любом сореве за < 5 минут обучения